In [ ]:
import pandas as pd
import helpers
import os

base_path = os.path.dirname(__file__)
data_path = os.path.join(base_path, "..", "data", "cleaned", "ds_jobs.csv")
df = pd.read_csv(data_path)

#standardizing job titles
df["title"] = df["title"].str.lower()
df["title"] = df["title"].str.replace("sr.", "senior")
df["title"] = df["title"].str.replace(r"\b(i|ii|iii)\b", "", regex=True)

df["job_type"] = df["title"].apply(helpers.job_type)
#extract years of experience from description
df["description"] = df["description"].str.lower()
df["years_experience"] = df["description"].apply(helpers.extract_experience)
#clean locations
df["location"] = df["location"].str.strip()
df[["part1", "part2", "part3"]] = df["location"].str.split(",", expand=True)
df["part1"] = df["part1"].str.strip()
df["part2"] = df["part2"].str.strip()
df["part3"] = df["part3"].str.strip()
df["state"] = df["part2"].fillna(df["part1"])
df.loc[df["state"] == "US", "state"] = None
df.loc[df["state"] == "Remote", "state"] = None
#removes remote jobs from map
df.loc[df["is_remote"] == True, "state"] = None
jobs_per_state = (
    df.groupby("state")
      .size()
      .reset_index(name="job_count")
)
#clean company size
df["company_size"] = df["company_num_employees"].str.replace("employees", "", case=False).str.strip()
df["company_size_numeric"] = df["company_size"].apply(helpers.company_size_clean)

#group by experience level
df["experience_level"] = df["years_experience"].apply(helpers.experience_group)

df = df.drop([
    "job_level", "job_function", "Unnamed: 0.1", "Unnamed: 0", "id", "job_url", "job_url_direct", "logo_photo_url", "banner_photo_url", "ceo_name", "ceo_photo_url",
    "salary_source", "interval", "emails", "company_url", "company_url_direct", "listing_type"], axis=1)
df = df[df.years_experience<41]
df.to_csv("data/cleaned/ds_jobs.csv", index=False)

In [3]:
df['company_size_numeric'].min()

1.0

In [4]:
df['company_size_numeric'].max()

10000.0

In [5]:
df.columns

Index(['site', 'title', 'company', 'location', 'job_type', 'date_posted',
       'min_amount', 'max_amount', 'currency', 'is_remote', 'company_industry',
       'description', 'company_addresses', 'company_num_employees',
       'company_revenue', 'company_description', 'mean_salary',
       'cleaned_description', 'years_experience', 'part1', 'part2', 'part3',
       'state', 'company_size', 'company_size_numeric', 'experience_level'],
      dtype='object')

In [6]:
df = pd.read_csv("data/raw/ds_jobs.csv")
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'id', 'site', 'job_url', 'job_url_direct',
       'title', 'company', 'location', 'job_type', 'date_posted',
       'salary_source', 'interval', 'min_amount', 'max_amount', 'currency',
       'is_remote', 'job_level', 'job_function', 'company_industry',
       'listing_type', 'emails', 'description', 'company_url',
       'company_url_direct', 'company_addresses', 'company_num_employees',
       'company_revenue', 'company_description', 'logo_photo_url',
       'banner_photo_url', 'ceo_name', 'ceo_photo_url', 'mean_salary',
       'cleaned_description'],
      dtype='object')